# UIT DSC 2026 LegalIR - Step 2 Chunk-BM25 Baseline

Reads Step 1 data artifacts from Kaggle Dataset and evaluates a dependency-free Chunk-BM25 document ranking baseline.

This step is CPU-bound; GPU is not required.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
PUBLIC_FILE = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json')
OUTPUT_DIR = Path('/kaggle/working/step2')

SCRIPT_CANDIDATES = [
    Path('/kaggle/working/legalir_step2_bm25.py'),
    Path('/kaggle/working/step2/legalir_step2_bm25.py'),
    Path('/kaggle/input/dscuit2026-code/task1/pipeline/step2/legalir_step2_bm25.py'),
    Path('/kaggle/input/dscuit2026/task1/pipeline/step2/legalir_step2_bm25.py'),
    Path('legalir_step2_bm25.py'),
    Path('task1/pipeline/step2/legalir_step2_bm25.py'),
]
SCRIPT_PATH = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)
if SCRIPT_PATH is None:
    raise FileNotFoundError('Cannot find legalir_step2_bm25.py. Add it as a Kaggle utility script or attach this repo as a dataset.')

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SCRIPT_PATH:', SCRIPT_PATH)
assert (DATA_ROOT / 'chunks.jsonl').exists()
assert (DATA_ROOT / 'train_split.json').exists()
assert (DATA_ROOT / 'dev_split.json').exists()
assert PUBLIC_FILE.exists()

## Optional public submission

Run this only when the dev result is worth spending one leaderboard submission attempt.

In [ ]:
MAKE_PUBLIC_SUBMISSION = False

if MAKE_PUBLIC_SUBMISSION:
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        '--data-root', str(DATA_ROOT),
        '--public-file', str(PUBLIC_FILE),
        '--output-dir', str(OUTPUT_DIR / 'public_run'),
        '--top-chunks', '300',
        '--top-docs', '100',
        '--evidence-per-doc', '3',
        '--k1', '1.5',
        '--b', '0.75',
        '--heading-weight', '2.0',
        '--mean-top3-weight', '0.20',
        '--support-weight', '0.05',
        '--predict-public',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
    print('Submission ZIP:', OUTPUT_DIR / 'public_run' / 'submission' / 'submission.zip')
else:
    print('Skipped public submission. Set MAKE_PUBLIC_SUBMISSION = True when ready.')

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--top-chunks', '300',
    '--top-docs', '100',
    '--evidence-per-doc', '3',
    '--k1', '1.5',
    '--b', '0.75',
    '--heading-weight', '2.0',
    '--mean-top3-weight', '0.20',
    '--support-weight', '0.05',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
report_path = OUTPUT_DIR / 'reports' / 'run_report.json'
metrics_path = OUTPUT_DIR / 'metrics' / 'dev_metrics.json'
with report_path.open('r', encoding='utf-8') as f:
    report = json.load(f)
with metrics_path.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

print(json.dumps({
    'config': report['config'],
    'index': report['index'],
    'dev_macro': metrics['macro'],
    'by_gold_count': metrics['by_gold_count'],
}, ensure_ascii=False, indent=2))

In [ ]:
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)